# FakeNewsDetection – Google Colab pipeline

**Futtatás előtt:** Runtime → Change runtime type → **T4 GPU**.

Az adatot a Kaggle-ről tölti le API-kulccsal (Kaggle → Settings → API → **Create New Token**, ez letölt egy `kaggle.json` fájlt).
Minden eredmény a **Google Drive**-odra kerül (`MyDrive/FakeNewsDetection`), így ha a Colab megszakad, az addigi munka megmarad,
és újraindítás után a már kész lépéseket (pl. az embeddingeket) átugorja.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/funnymrmagyar/FakeNewsDetection.git /content/repo || (cd /content/repo && git pull -q)
%cd /content/repo
!git log --oneline -1
!pip install -q timm shap tabulate kaggle

In [ ]:
# Kaggle API-kulcs: válaszd ki a letöltött kaggle.json fájlt
import os
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(up[next(iter(up))])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json rendben')

In [ ]:
import glob, os
DRIVE = '/content/drive/MyDrive/FakeNewsDetection'
DATA, EMB, OUT = f'{DRIVE}/data', f'{DRIVE}/emb', f'{DRIVE}/results'
os.makedirs(DRIVE, exist_ok=True)
SPLITS = f'{EMB}/splits.csv'
REUSE = os.path.exists(f'{EMB}/text_bert.npy') and os.path.exists(SPLITS)

# Az adat letöltése a Colab saját lemezére (gyorsabb, mint a Drive; ~5 GB, 5-10 perc)
IN = '/content/input'
if not glob.glob(f'{IN}/**/multimodal_train.tsv', recursive=True):
    !kaggle datasets download -d vanshikavmittal/fakeddit-dataset -p {IN}/tsv --unzip -q
if not glob.glob(f'{IN}/**/fakeddit_small_img', recursive=True):
    !kaggle datasets download -d rajatdhanuka/fakeddit-50k -p {IN}/img --unzip -q
TSV_DIR = os.path.dirname(glob.glob(f'{IN}/**/multimodal_train.tsv', recursive=True)[0])
IMG_DIR = [d for d in glob.glob(f'{IN}/**/fakeddit_small_img', recursive=True) if os.path.isdir(d)][0]
print('REUSE (kész embeddingek a Drive-on)' if REUSE else 'ÚJ FUTÁS', TSV_DIR, IMG_DIR, sep='\n')

## 0. fázis – adat, címkék, felosztás
Ellenőrizd a **címkekódolást** és a stats-táblát, és küldd el, mielőtt továbbmész.

In [ ]:
if not REUSE:
    !python -m src.data.prepare --tsv-dir "$TSV_DIR" --img-dir "$IMG_DIR" --out "$DATA"
import pandas as pd
display(pd.read_csv(f'{DATA}/stats.csv'))

## 1. fázis – embeddingek (20–40 perc, egyszer; a Drive-ra mentődnek)

In [ ]:
if not REUSE:
    !python -m src.data.extract --splits "$DATA/splits.csv" --out "$EMB" --which text_bert img_vit img_resnet18 --batch-size 64

## 2. fázis – a mag: M1–M6

In [ ]:
common = f'--emb-dir "{EMB}" --splits "{SPLITS}" --out "{OUT}"'
!python -m src.training.train --name M1_text        --features text_bert               {common}
!python -m src.training.train --name M2_resnet      --features img_resnet18            {common}
!python -m src.training.train --name M3_vit         --features img_vit                 {common}
!python -m src.eval.late_fusion --a M1_text --b M2_resnet --name M4_late_fusion --out "{OUT}"
!python -m src.training.train --name M5_bert_resnet --features text_bert img_resnet18  {common}
!python -m src.training.train --name M6_bert_vit    --features text_bert img_vit       {common}
!python -m src.eval.report --out "$OUT"
!python -m src.stats.significance --out "$OUT" --pairs M3_vit:M2_resnet M6_bert_vit:M1_text M6_bert_vit:M5_bert_resnet M6_bert_vit:M4_late_fusion

## 3. fázis – extrák (BLIP 1–2 óra)
Ha a Colab közben megszakad: futtasd újra az első 3 kódcellát, majd ezt. A `captions` kinyerés elölről kezdődik, de a többi megmarad.

In [ ]:
if not os.path.exists(f'{EMB}/caption_bert.npy'):
    !python -m src.data.extract --splits "$SPLITS" --out "$EMB" --which caption_bert
if not os.path.exists(f'{EMB}/clip_score.npy'):
    !python -m src.data.extract --splits "$SPLITS" --out "$EMB" --which clip_score

In [ ]:
common = f'--emb-dir "{EMB}" --splits "{SPLITS}" --out "{OUT}"'
!python -m src.training.train --name M7_bert_blip_vit      --features text_bert img_vit caption_bert            {common}
!python -m src.training.train --name M8_bert_blip_vit_clip --features text_bert img_vit caption_bert clip_score {common}
!python -m src.training.train --name A1_no_text  --features img_vit caption_bert   {common}
!python -m src.training.train --name A2_no_image --features text_bert caption_bert {common}

## 4. Eredmények: táblák, szignifikancia, SHAP, ábrák

In [ ]:
!python -m src.eval.report --out "$OUT" --roc M1_text M3_vit M5_bert_resnet M6_bert_vit M7_bert_blip_vit --cm M6_bert_vit M7_bert_blip_vit
!python -m src.stats.significance --out "$OUT" --pairs M3_vit:M2_resnet M6_bert_vit:M1_text M6_bert_vit:M5_bert_resnet M6_bert_vit:M4_late_fusion M7_bert_blip_vit:M6_bert_vit M8_bert_blip_vit_clip:M7_bert_blip_vit
!python -m src.explain.shap_analysis --ckpt "$OUT/models/M7_bert_blip_vit_seed42.pt" --emb-dir "$EMB" --splits "$SPLITS" --out "$OUT" --per-dim
import pandas as pd
from IPython.display import Image as Img
display(pd.read_csv(f'{OUT}/table_main.csv'), pd.read_csv(f'{OUT}/significance.csv'))
display(Img(f'{OUT}/shap/M7_bert_blip_vit_modality.png'), Img(f'{OUT}/roc.png'))

## 5. Export a webappnak
A `fusion_head.pt` és a `results_export.zip` a Drive-odon lesz (`MyDrive/FakeNewsDetection`).
Ha M7-et nem futtattad, írd át `M6_bert_vit`-re.

In [ ]:
import pandas as pd, shutil
MODEL = 'M7_bert_blip_vit'
s = pd.read_csv(f'{OUT}/summary.csv').drop_duplicates(['name', 'seed'], keep='last')
best = s[s.name == MODEL].sort_values('val_f1', ascending=False).iloc[0]   # val alapján, nem teszt alapján!
shutil.copy(f"{OUT}/models/{MODEL}_seed{int(best.seed)}.pt", f'{DRIVE}/fusion_head.pt')
shutil.make_archive(f'{DRIVE}/results_export', 'zip', OUT, '.')
print(f'fusion_head.pt <- {MODEL} seed {int(best.seed)} (val F1 {best.val_f1:.4f}, test F1 {best.f1:.4f})')